In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib as m
from tqdm import tqdm
from scipy import linalg
import scipy as sp
import os
import pickle
import glob
import h5py
import random
import matplotlib.cm as cm

from scipy.constants import *

phi0 = physical_constants["mag. flux quantum"][0]
eps0 = epsilon_0

from scipy.special import eval_hermite
import scfitpy as Qfit

In [ ]:
f = open("4JJUS_PT3_1.pickle", "rb")  # fitUSCSc2
fitparams = pickle.load(f)
Ej = fitparams[0] * 1e2
Ec = fitparams[1]
Lr = fitparams[2] * 1e-9
Cr = fitparams[3] * 1e-13
params0 = [Ej, Ec, Lr, Cr, fitparams[4], fitparams[5]]

In [ ]:
pst = 31
phi_list = np.linspace(0.49, 0.5, pst)
nl = [6, 7, 8, 9]

In [ ]:
E = [[[[] for i in range(len(nl))] for i in range(len(nl))] for i in range(len(nl))]
for i, n1 in enumerate(nl):
    for j, n2 in enumerate(nl):
        for k, n3 in enumerate(nl):
            E[i][j][k] = Qfit.circuit_spectrum_Q(phi_list, params0, [n1, n2, n3])

In [ ]:
with open("E_space.pickle", mode="wb") as fo:
    pickle.dump(E, fo)

In [ ]:
f = open("E_space.pickle", "rb")
E = pickle.load(f)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plt.rcParams["font.size"] = 19
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"

for i in range(len(nl)):
    for j in range(len(nl)):
        for k in range(len(nl)):
            ax.plot(
                phi_list,
                np.array(E[i][j][k][1]) - np.array(E[i][j][k][0]),
                color=cm.hsv((i + j + k) / 15.0),
                label=str(i + min(nl)) + str(j + min(nl)) + str(k + min(nl)),
            )

plt.grid()
plt.xlabel(r"$\varphi_{ex}/2\pi$[a.u.]", fontsize=24)
plt.ylabel(r"$E_{01}/h\ $[GHz]", fontsize=22)
plt.savefig("QspaceE10_0.png", dpi=700)
plt.show()

In [ ]:
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(8, 7))
plt.rcParams["font.size"] = 19
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
nm = len(nl) - 1

custom_colors = ["cyan", "red", "blue"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=custom_colors)

cmap = cm.YlOrBr
norm = mcolors.Normalize(vmin=18, vmax=27)

combinations = [
    (i, j, k) for i in range(len(nl)) for j in range(len(nl)) for k in range(len(nl))
]
combinations.sort(key=lambda x: x[0] + x[1] + x[2])

for i, j, k in combinations:
    ed01 = abs(
        np.array(E[i][j][k][1])
        - np.array(E[i][j][k][0])
        - (np.array(E[nm][nm][nm][1]) - np.array(E[nm][nm][nm][0]))
    )
    ax.plot(phi_list, ed01, color=cm.YlOrBr((i + j + k) / (27 - 18)))
for i, j, k in combinations:
    ed01 = abs(
        np.array(E[i][j][k][1])
        - np.array(E[i][j][k][0])
        - (np.array(E[nm][nm][nm][1]) - np.array(E[nm][nm][nm][0]))
    )
    if max(ed01) < 2e-3 and i + j + k < 3 * len(nl) - 6:
        plt.plot(
            phi_list,
            ed01,
            "--",
            lw=3,
            label=str(min(nl) + i) + str(min(nl) + j) + str(min(nl) + k),
        )

ax.set_xlim([0.49, 0.5])
plt.grid()
plt.yscale("log")
plt.xlabel(r"$\varphi_{ex}/2\pi$[a.u.]", fontsize=24)
plt.ylabel(r"$E_{01}(n,m,l)/h-E_{01}(9,9,9)/h\ $[GHz]", fontsize=22)
plt.legend(loc="lower left", ncol=1)

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"Charge space $n+l+m$")

plt.savefig("QspaceE10.png", dpi=700)
plt.show()

In [ ]:
ES = []
qs_list = [int(i) for i in np.linspace(2, 25, 24)]
Ev, Ek = Qfit.circuit_spectrum_Qket(phi_list, params0, [9, 6, 7])

In [ ]:
for qs in qs_list:
    EQS = Qfit.circuit_spectrum_QR(Ev, Ek, params0, [9, 6, 7], qs)
    ES.append(EQS)

In [ ]:
with open("ES_space.pickle", mode="wb") as fo:
    pickle.dump(ES, fo)

In [ ]:
f = open("ES_space.pickle", "rb")
E = pickle.load(f)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plt.rcParams["font.size"] = 19
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"

cmap = cm.hsv
norm = mcolors.Normalize(vmin=2, vmax=25)


for i in range(len(qs_list)):
    ax.plot(
        phi_list,
        np.array(ES[i][1]) - np.array(ES[i][0]),
        "-.",
        color=cm.hsv(i / 25.0),
        label=str(i + min(nl)) + str(j + min(nl)) + str(k + min(nl)),
    )
    ax.plot(
        phi_list,
        np.array(ES[i][2]) - np.array(ES[i][0]),
        "--",
        color=cm.hsv(i / 25.0),
        label=str(i + min(nl)) + str(j + min(nl)) + str(k + min(nl)),
    )
    ax.plot(
        phi_list,
        np.array(ES[i][3]) - np.array(ES[i][1]),
        "-",
        color=cm.hsv(i / 25.0),
        label=str(i + min(nl)) + str(j + min(nl)) + str(k + min(nl)),
    )

ax.set_ylim([5, 5.6])
ax.set_xlim([0.49, 0.5])
plt.grid()
plt.xlabel(r"$\varphi_{ex}/2\pi$[a.u.]", fontsize=24)
plt.ylabel(r"$\omega_{31}/2\pi$[GHz]", fontsize=22)

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Qubit space")

plt.savefig("qspace_w31.png", bbox_inches="tight", pad_inches=0.5, dpi=700)
plt.show()

In [ ]:
r2 = []
for i in range(len(qs_list)):
    r0 = (
        abs(
            np.array(ES[i][1])
            - np.array(ES[i][0])
            - np.array(ES[len(qs_list) - 1][1])
            + np.array(ES[len(qs_list) - 1][0])
        )
        + abs(
            np.array(ES[i][2])
            - np.array(ES[i][0])
            - np.array(ES[len(qs_list) - 1][2])
            + np.array(ES[len(qs_list) - 1][0])
        )
        + abs(
            np.array(ES[i][3])
            - np.array(ES[i][1])
            - np.array(ES[len(qs_list) - 1][3])
            + np.array(ES[len(qs_list) - 1][1])
        )
    )
    r2.append(max(r0))

In [ ]:
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(figsize=(8, 6))
plt.rcParams["font.size"] = 25
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
ax.plot(qs_list, np.array(r2), "-*")

ax.set_xlim([2, 25])
ax.xaxis.set_major_locator(MaxNLocator(5))
plt.yscale("log")
plt.grid()
plt.xlabel(r"Qubit space", fontsize=24)
plt.ylabel(r"Maximum energy difference [GHz]", fontsize=22)

plt.savefig("Qspace.png", bbox_inches="tight", pad_inches=0.5, dpi=700)
plt.show()